# Import Library

In [9]:
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import glob
import os
import re
from tqdm import tqdm # loading bar
from transformers import pipeline
from dotenv import load_dotenv
import requests
import json
import time

pd.set_option('display.max_colwidth', None)

# Setup and Data Preparation

In [2]:
# read data 
df_data_final = pd.read_csv(
    'tweets-data/data_gabungan_pati_final.csv', 
    dtype={
        'id': str,
        'conversation_id': str,
        'source_tweet_id': str,
        'author': str,
        'reply_to': str
    }
)

# check result
print(df_data_final[['id', 'source_tweet_id']].head())
print("\nInfo Tipe Data:")
print(df_data_final.dtypes)

                    id      source_tweet_id
0  2013214486900154533  2013214486900154533
1  2013210519931654580  2013210519931654580
2  2013210033648234543  2013210033648234543
3  2013474973415776705  2013474973415776705
4  2013978983533305858  2013978983533305858

Info Tipe Data:
id                 object
conversation_id    object
source_tweet_id    object
relation_type      object
text               object
author             object
author_location    object
created_at         object
like_count          int64
reply_count         int64
retweet_count       int64
quote_count         int64
view_count          int64
url                object
lang               object
reply_to           object
hashtags           object
dtype: object


In [4]:
df_data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2386 entries, 0 to 2385
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               2386 non-null   object
 1   conversation_id  2386 non-null   object
 2   source_tweet_id  1117 non-null   object
 3   relation_type    2386 non-null   object
 4   text             2386 non-null   object
 5   author           2386 non-null   object
 6   author_location  1449 non-null   object
 7   created_at       2386 non-null   object
 8   like_count       2386 non-null   int64 
 9   reply_count      2386 non-null   int64 
 10  retweet_count    2386 non-null   int64 
 11  quote_count      2386 non-null   int64 
 12  view_count       2386 non-null   int64 
 13  url              2386 non-null   object
 14  lang             2386 non-null   object
 15  reply_to         1039 non-null   object
 16  hashtags         2386 non-null   object
dtypes: int64(5), object(12)
memory us

In [5]:
pretrained_model = 'ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa'
sentiment_pipeline = pipeline(
    'sentiment-analysis',
    model=pretrained_model,
    tokenizer=pretrained_model
)

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 408.04it/s, Materializing param=classifier.weight]                                      
BertForSequenceClassification LOAD REPORT from: ayameRushia/bert-base-indonesian-1.5G-sentiment-analysis-smsa
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
def clean_text(text):
    # lowercase
    text = str(text).lower()
    # delete URL & Mention
    text = re.sub(r'http\S+', '', text)
    text = re.sub(r'@[A-Za-z0-9_]+', '', text)
    # remove only # 
    text = re.sub(r'#', '', text)
    # remove special character 
    text = re.sub(r'[^a-zA-Z0-9\s.,!?\'"]', '', text)
    # remove whitespace
    text = re.sub(r'\s+', ' ', text)
    
    return text.strip()

def extract_mentions(text):
    mentions = re.findall(r'@([A-Za-z0-9_]+)', str(text))
    return mentions

# Sentiment Classification

In [7]:
df_data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2386 entries, 0 to 2385
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               2386 non-null   object
 1   conversation_id  2386 non-null   object
 2   source_tweet_id  1117 non-null   object
 3   relation_type    2386 non-null   object
 4   text             2386 non-null   object
 5   author           2386 non-null   object
 6   author_location  1449 non-null   object
 7   created_at       2386 non-null   object
 8   like_count       2386 non-null   int64 
 9   reply_count      2386 non-null   int64 
 10  retweet_count    2386 non-null   int64 
 11  quote_count      2386 non-null   int64 
 12  view_count       2386 non-null   int64 
 13  url              2386 non-null   object
 14  lang             2386 non-null   object
 15  reply_to         1039 non-null   object
 16  hashtags         2386 non-null   object
dtypes: int64(5), object(12)
memory us

In [8]:
df_data_final[['id', 'conversation_id']].head()

,id,conversation_id
0,2013214486900154533,2013214486900154533
1,2013210519931654580,2013210519931654580
2,2013210033648234543,2013210033648234543
3,2013474973415776705,2013474973415776705
4,2013978983533305858,2013978983533305858


In [9]:
df_data_final['relation_type'].value_counts()

relation_type
search_result    1269
reply             857
quote             252
main_tweet          8
Name: count, dtype: int64

In [10]:
df_data_final['reply_to'].value_counts()

reply_to
jhonsitorus_19    197
kumparan          178
Heraloebss        107
kompascom         107
Urrangawak         64
                 ... 
apluswire           1
Oxair_19            1
comeback378         1
IdaOno70639         1
ldwx1454            1
Name: count, Length: 186, dtype: int64

In [11]:
df_data_final[df_data_final['relation_type']=='search_result']['reply_to'].sample(5, random_state=1)

2110        NaN
1318        NaN
1711        NaN
1492    Osky_AM
2280        NaN
Name: reply_to, dtype: object

In [12]:
df_data_final[df_data_final['relation_type']=='quote']['reply_to'].sample(5)

485     NaN
408     NaN
452     NaN
451     NaN
1109    NaN
Name: reply_to, dtype: object

In [13]:
df_data_final[df_data_final['relation_type']=='quote']['reply_to'].isnull().sum()

np.int64(247)

In [14]:
df_data_final[df_data_final['relation_type']=='reply']['reply_to'].isnull().sum()

np.int64(0)

In [15]:
# Cek sampel data Quote yang reply_to-nya NaN
cek_quote = df_data_final[
    (df_data_final['relation_type'] == 'quote') & 
    (df_data_final['reply_to'].isna())
][['text', 'url', 'source_tweet_id']]


display(cek_quote.head(5))

,text,url,source_tweet_id
121,God u make me ashamed bruhh,https://x.com/mushybouquet/status/2013674648857387041,2013214486900154533
122,kenapa kota yang gua tinggali pemimpinnya ga pernah bener,https://x.com/bayunarr_/status/2013664417368416443,2013214486900154533
123,Syukurrr dan Hukum yg berat @KPK_RI !,https://x.com/AntoniGirsang/status/2013647822562979956,2013214486900154533
124,HAHAHAHWHAHAHAHAHAHAHAHHAHAHAAHAHHAHHAHHAHHAHAHHAHHAHAHAHAHHAHHAHHAHAHHAHAHAHHAAHAHHAHHAHHAHAHHAHAHAHHAHAJHAJHAHAH,https://x.com/vanroem/status/2013608510366400770,2013214486900154533
125,Bupati ndlogok,https://x.com/syafeeee/status/2013568826848485658,2013214486900154533


In [16]:
df_data_final['source_tweet_id'].sample(5)

776     2013240346801561820
1503                    NaN
686     2013240346801561820
1365                    NaN
1675                    NaN
Name: source_tweet_id, dtype: object

In [17]:
# fill quote interaction
id_to_author_map = df_data_final.set_index('id')['author'].to_dict()

# fin the target with source twit id
def fix_quote_target(row):
    # if reply to is there, then return
    if pd.notna(row['reply_to']):
        return row['reply_to']
    
    # make sure having source tweet id
    if row['relation_type'] == 'quote' and pd.notna(row['source_tweet_id']):
        source_id = row['source_tweet_id']
        
        # find source tweet with id then get the userna,e
        found_author = id_to_author_map.get(source_id)
        
        return found_author # return username
    
    return None

# fixed reply to with another column (add new column)
print("Memperbaiki target Quote Tweet...")
df_data_final['reply_to_fixed'] = df_data_final.apply(fix_quote_target, axis=1)

# fill na with value from a new column
df_data_final['reply_to'] = df_data_final['reply_to'].fillna(df_data_final['reply_to_fixed'])

# check the output (sanity check)
print("\nStatistik Penyelamatan:")
print(df_data_final[df_data_final['relation_type']=='quote'][['reply_to']].info())

Memperbaiki target Quote Tweet...

Statistik Penyelamatan:
<class 'pandas.core.frame.DataFrame'>
Index: 252 entries, 121 to 1116
Data columns (total 1 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   reply_to  252 non-null    object
dtypes: object(1)
memory usage: 3.9+ KB
None


In [19]:
df_data_final[df_data_final['relation_type']=='quote']['reply_to'].isnull().sum()

np.int64(0)

In [20]:
df_data_final[df_data_final['relation_type']=='quote'][['reply_to', 'reply_to_fixed']].sample(10)

,reply_to,reply_to_fixed
128,kompascom,kompascom
121,kompascom,kompascom
180,callmeelily,callmeelily
845,jhonsitorus_19,jhonsitorus_19
405,kumparan,kumparan
843,jhonsitorus_19,jhonsitorus_19
499,kumparan,kumparan
854,jhonsitorus_19,jhonsitorus_19
561,MurtadhaOne1,MurtadhaOne1
489,kumparan,kumparan


it's already fixed

In [21]:
df_data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2386 entries, 0 to 2385
Data columns (total 18 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               2386 non-null   object
 1   conversation_id  2386 non-null   object
 2   source_tweet_id  1117 non-null   object
 3   relation_type    2386 non-null   object
 4   text             2386 non-null   object
 5   author           2386 non-null   object
 6   author_location  1449 non-null   object
 7   created_at       2386 non-null   object
 8   like_count       2386 non-null   int64 
 9   reply_count      2386 non-null   int64 
 10  retweet_count    2386 non-null   int64 
 11  quote_count      2386 non-null   int64 
 12  view_count       2386 non-null   int64 
 13  url              2386 non-null   object
 14  lang             2386 non-null   object
 15  reply_to         1286 non-null   object
 16  hashtags         2386 non-null   object
 17  reply_to_fixed   1286 non-null   

In [22]:
# drop a new column (column helper)
df_data_final.drop(columns=['reply_to_fixed'], inplace=True)

# sanity check
df_data_final.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2386 entries, 0 to 2385
Data columns (total 17 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   id               2386 non-null   object
 1   conversation_id  2386 non-null   object
 2   source_tweet_id  1117 non-null   object
 3   relation_type    2386 non-null   object
 4   text             2386 non-null   object
 5   author           2386 non-null   object
 6   author_location  1449 non-null   object
 7   created_at       2386 non-null   object
 8   like_count       2386 non-null   int64 
 9   reply_count      2386 non-null   int64 
 10  retweet_count    2386 non-null   int64 
 11  quote_count      2386 non-null   int64 
 12  view_count       2386 non-null   int64 
 13  url              2386 non-null   object
 14  lang             2386 non-null   object
 15  reply_to         1286 non-null   object
 16  hashtags         2386 non-null   object
dtypes: int64(5), object(12)
memory us

## Create Label Sentiment

In [23]:
# progress bar
tqdm.pandas()

# cleaning text
print("[*] Applying text cleaning...")
df_data_final['clean_text'] = df_data_final['text'].progress_apply(clean_text)

# sentiment analysis for all tweets
print("[*] Running Sentiment Analysis for all tweets")

# wrapper function
def get_sentiment(text):
    if not text: return 'NEUTRAL', 0.0
    try:
        # truncation to prevent error (text long)
        res = sentiment_pipeline(text, truncation=True)[0] 
        return res['label'], res['score']
    except:
        return 'NEUTRAL', 0.0


df_data_final[['sentiment', 'score']] = df_data_final['clean_text'].progress_apply(
    lambda x: pd.Series(get_sentiment(x))
)

# check the output
print("\nDistribution of Sentiment:")
print(df_data_final['sentiment'].value_counts())

[*] Applying text cleaning...


100%|██████████| 2386/2386 [00:00<00:00, 23272.37it/s]


[*] Running Sentiment Analysis for all tweets


100%|██████████| 2386/2386 [02:36<00:00, 15.22it/s]



Distribution of Sentiment:
sentiment
Neutral     1379
Negative     759
Positive     180
NEUTRAL       68
Name: count, dtype: int64


In [42]:
df_data_final[['clean_text', 'sentiment']].sample(5, random_state=2)

,clean_text,sentiment
659,gak enak sih,Negative
2251,"usai diperiksa kpk di polres kudus, bupati pati sudewo dibawa ke semarang",Neutral
644,karma,Negative
308,makanya nurut sama atasan kalau turun ya turun malah kekeh mempertahanin jabatan,Negative
1166,"pdi perjuangan adl satu2nya partai yang ngotot memakzulkan bupati pati, sudewo. partai lain seperti gerindra, golkar, ppp, pkb, demokrat dan pks mati2an mempertahankan sudewo agar tidak lengser. pada akhirnya, kebenaranlah yang menang. satyam eva jayate, sudewo ditangkap kpk.",Neutral


## Sentiment with LLM

In [46]:
from dotenv import load_dotenv
import requests
import json
import time

In [49]:
# configuration
BATCH_SIZE = 10  # safe size
SAVE_EVERY = 5   # save every 50 data
OUTPUT_DIR = "output-data" 
OUTPUT_FILE = f"{OUTPUT_DIR}/bupati_pati_data_with_sentiment.csv" 
TEMP_DIR = f"{OUTPUT_DIR}/temp"
TEMP_FILE = f"{TEMP_DIR}/temp_sentiment_bupati_pati.csv"
MODEL_ID = "x-ai/grok-4.1-fast"

# create directory if not exists
if not os.path.exists(OUTPUT_DIR):
    os.makedirs(OUTPUT_DIR)
if not os.path.exists(TEMP_DIR):
    os.makedirs(TEMP_DIR)

# load api key
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
URL = "https://openrouter.ai/api/v1/chat/completions"

# prepare data 
df_process = df_data_final.copy()

# ensure id is string 
df_process['id'] = df_process['id'].astype(str)

print(f"[*] Starting sentiment & keyword extraction with Grok. Total data: {len(df_process)}")

# prompt for llm (detailed sentiment + keyword extraction)
system_prompt_final = """
ROLE:
You are a Senior Political Analyst & Social Media Expert specializing in Indonesian public sentiment regarding corruption and law enforcement, specifically the arrest of the Regent of Pati (Sudewo).

CONTEXT:
You are analyzing tweets about the arrest of the Regent of Pati (Sudewo) in a corruption case (OTT).
Indonesian netizens often use:
1. Javanese Language & Slang (e.g., "modyar", "rasain", "ndlogok").
2. Sarcasm (e.g., "Prestasinya membanggakan" meaning corruption).

INSTRUCTION:
Analyze each tweet to determine the sentiment and extract key topics/entities for a WordCloud.

OUTPUT FIELDS:
1. "sentiment":
   - POSITIVE: 
     * Expressing happiness/satisfaction over the arrest (e.g., "Alhamdulillah kena juga", "Akhirnya!").
     * Supporting the KPK/Law enforcement.
     * Condemning the corrupt behavior (e.g., "Mampus lo koruptor").
     * Sarcasm that implies happiness about the regent's downfall.
     * [EXPERT JUDGMENT]: Any other expression that aligns with supporting the arrest, justice process, or anti-corruption efforts.
   
   - NEGATIVE: 
     * Defending the regent (e.g., "Bapak orang baik", "Ini fitnah politik").
     * Expressing anger towards the central government/law enforcement (NOT towards the regent).
     * Acute Pessimism/Skepticism (e.g., "Halah paling nanti bebas lagi", "Hukum tumpul").
     * Feeling ashamed/sad about the region's reputation.
     * [EXPERT JUDGMENT]: Any other expression reflecting dissatisfaction, distrust, resistance, or critical views regarding the situation.

   - NEUTRAL: 
     * Objective news/Information sharing without opinion.
     * Unrelated to the specific case.

2. "keywords": 
   - Extract 5-8 MAIN WORDS/PHRASES (or fewer if the tweet is short) from the tweet that represent the topic.
   - Focus on:
     * Entities & Political Actors (e.g., "KPK", "Sudewo", "Partai", "DPRD")
     * Actions (e.g., "Tangkap", "Hukum Mati", "OTT")
     * Emotions (e.g., "Malu", "Kapok", "Geram")
     * Key Concepts/Objects (e.g., "Uang Rakyat", "Korupsi", "Pilkada", "Jalan Rusak").
     * [EXPERT JUDGMENT]: Other significant terms or slang that provide important context to the political narrative.
   - Normalize to lowercase and standard Indonesian if possible (e.g., convert "maling2" to "maling").
   - Format: A single string separated by comma (e.g., "korupsi, kpk, pati, hukum mati").

FORMAT OUTPUT (JSON LIST):
[
  {"id": "...", "sentiment": "...", "keywords": "..."},
  ...
]

NOTES:
- Be vigilant for SARCASM. Do not take apparent praise at face value.
- For keywords, keep it substantive. Ignore stopwords.
- If tweet is empty, return "NEUTRAL" and empty keywords.
"""

# create request function
def analyze_batch(batch_data):
    try:
        payload = {
            "model": MODEL_ID,
            "messages": [
                {"role": "system", "content": system_prompt_final},
                {"role": "user", "content": json.dumps(batch_data)}
            ],
            # grok parameters
            "temperature": 0.3, 
            "max_tokens": 2000,
            "reasoning": {"enabled": True} # enable reasoning for better accuracy
        }
        
        response = requests.post(
            url=URL,
            headers={
                "Authorization": f"Bearer {API_KEY}",
                "Content-Type": "application/json",
                "HTTP-Referer": "http://localhost:8501", 
                "X-Title": "SNA Pati Analysis" 
            },
            data=json.dumps(payload),
            timeout=180 
        )
        
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content']
            # remove markdown json if exists
            clean_content = content.replace("```json", "").replace("```", "").strip()
            return json.loads(clean_content)
        else:
            print(f"[!] API Error: {response.status_code} - {response.text}")
            return []
            
    except Exception as e:
        print(f"[!] Exception: {e}")
        return []
    
# execute looping
final_results = [] 
total_data = len(df_process)

# using tqdm
for i in tqdm(range(0, total_data, BATCH_SIZE)):
    # use 'text' (raw) to capture context
    batch_df = df_process.iloc[i : i + BATCH_SIZE][['id', 'text']]
    
    # handle nan
    batch_df['text'] = batch_df['text'].fillna("")
    
    # convert to dict
    batch_input = batch_df.to_dict(orient='records')
    
    # send to ai
    batch_result = analyze_batch(batch_input)
    
    if batch_result:
        # validate structure
        if isinstance(batch_result, list):
            final_results.extend(batch_result)
        elif isinstance(batch_result, dict):
             # check if result is wrapped in a dict key (e.g. 'tweets': [...])
             # simple fallback: look for the first list value found
             found_list = False
             for key, value in batch_result.items():
                 if isinstance(value, list):
                     final_results.extend(value)
                     found_list = True
                     break
             
             # if no list found, maybe the dict IS the single result
             if not found_list:
                 final_results.append(batch_result)

        # save temp
        if (i // BATCH_SIZE) % SAVE_EVERY == 0:
            pd.DataFrame(final_results).to_csv(TEMP_FILE, index=False)
    
    # sleep to respect rate limit
    time.sleep(3) 
    

# merge & save
print("[*] Analysis done. Merging results...")
df_llm = pd.DataFrame(final_results)

if not df_llm.empty:
    # ensure id is string
    df_llm['id'] = df_llm['id'].astype(str)
    
    # rename columns (keywords replaced reasoning)
    df_llm.rename(columns={'sentiment': 'sentiment_llm', 'keywords': 'sentiment_keywords'}, inplace=True)

    # merge data (left join)
    df_final_result = pd.merge(df_process, df_llm, on='id', how='left')
    
    # save final
    df_final_result.to_csv(OUTPUT_FILE, index=False)
    print(f"[*] Final file saved: {OUTPUT_FILE}")
    
    # preview
    if 'sentiment_llm' in df_final_result.columns:
        print(df_final_result[['text', 'sentiment_llm', 'sentiment_keywords']].sample(5))
    else:
        print("[!] Column 'sentiment_llm' not found. Check merge process.")
else:
    print("[!] Failed to get results (Empty List). Check API.")

[*] Starting sentiment & keyword extraction with Grok. Total data: 2386


 16%|█▋        | 39/239 [09:34<48:06, 14.43s/it]  

[!] Exception: Expecting value: line 1 column 1 (char 0)


 92%|█████████▏| 220/239 [1:01:59<04:20, 13.69s/it]

[!] API Error: 401 - {"error":{"message":"User not found.","code":401}}


100%|██████████| 239/239 [1:05:55<00:00, 16.55s/it]


[*] Analysis done. Merging results...
[*] Final file saved: output-data/bupati_pati_data_with_sentiment.csv
                                                                                                                                                                                                text  \
6                                                                                                 Akhirnya Bupati Pati Sudewo yang arogan tumbang, warga rayakan kemenangan. https://t.co/b98WSI4cXa   
1511                                                                                                                       Pasca OTT Bupati Sudewo Warga Pati Gelar Syukuran https://t.co/2D885AIH9Y   
560                                                                                                                                                                    gawe nekakno kabah iku paling   
564                                                                                         

In [51]:
df_llm.isnull().sum()

id                    0
sentiment_llm         0
sentiment_keywords    0
dtype: int64

In [7]:
df_llm = pd.read_csv(
    'output-data/temp/temp_sentiment_bupati_pati.csv', 
    dtype={'id': str})

df_process = pd.read_csv(
    'output-data/bupati_pati_data_with_sentiment.csv', 
    dtype={'id': str})



In [8]:
print(f"Total Data Asli: {len(df_process)}")
print(f"Total Data Berhasil dari LLM: {len(df_llm)}")

all_ids = set(df_process['id'].astype(str))
success_ids = set(df_llm['id'].astype(str))

missing_ids = all_ids - success_ids
print(f"Jumlah ID yang Gagal/Hilang: {len(missing_ids)}")

# Cek detail yang hilang
if len(missing_ids) > 0:
    print("\n[!] ID yang hilang (Sample 5):")
    print(list(missing_ids)[:5])
    
    df_missing = df_process[df_process['id'].astype(str).isin(missing_ids)]
    print("\nTweet yang gagal dianalisis:")
    print(df_missing[['id', 'text']].head())
else:
    print("\nSemua data lengkap")

Total Data Asli: 2386
Total Data Berhasil dari LLM: 2340
Jumlah ID yang Gagal/Hilang: 46

[!] ID yang hilang (Sample 5):
['2013214541476254077', '2013130703945241008', '2013187315204776039', '2013433781370405282', '2013440845715026365']

Tweet yang gagal dianalisis:
                      id  \
390  2013232539817934957   
391  2013232239010615738   
392  2013231740899537364   
393  2013231295921340646   
394  2013231044049437045   

                                                                                                                                                                                                                       text  
390                                                                                                                                                               @kumparan Waaahhh…hari ini ada 2 OTT ya,Madiun sama Pati?  
391                                                                                                                     

Recover data

In [10]:
# configuration
load_dotenv()
API_KEY = os.getenv("OPENROUTER_API_KEY")
URL = "https://openrouter.ai/api/v1/chat/completions"
MODEL_ID = "x-ai/grok-4.1-fast"

# load data
df_llm = pd.read_csv('output-data/temp/temp_sentiment_bupati_pati.csv', dtype={'id': str})
df_process = pd.read_csv('output-data/bupati_pati_data_with_sentiment.csv', dtype={'id': str})

# system prompt
system_prompt_final = """
ROLE:
You are a Senior Political Analyst & Social Media Expert specializing in Indonesian public sentiment regarding corruption and law enforcement, specifically the arrest of the Regent of Pati (Sudewo).

CONTEXT:
You are analyzing tweets about the arrest of the Regent of Pati (Sudewo) in a corruption case (OTT).
Indonesian netizens often use:
1. Javanese Language & Slang (e.g., "modyar", "rasain", "ndlogok").
2. Sarcasm (e.g., "Prestasinya membanggakan" meaning corruption).

INSTRUCTION:
Analyze each tweet to determine the sentiment and extract key topics/entities for a WordCloud.

OUTPUT FIELDS:
1. "sentiment":
   - POSITIVE: 
     * Expressing happiness/satisfaction over the arrest (e.g., "Alhamdulillah kena juga", "Akhirnya!").
     * Supporting the KPK/Law enforcement.
     * Condemning the corrupt behavior (e.g., "Mampus lo koruptor").
     * Sarcasm that implies happiness about the regent's downfall.
     * [EXPERT JUDGMENT]: Any other expression that aligns with supporting the arrest, justice process, or anti-corruption efforts.
   
   - NEGATIVE: 
     * Defending the regent (e.g., "Bapak orang baik", "Ini fitnah politik").
     * Expressing anger towards the central government/law enforcement (NOT towards the regent).
     * Acute Pessimism/Skepticism (e.g., "Halah paling nanti bebas lagi", "Hukum tumpul").
     * Feeling ashamed/sad about the region's reputation.
     * [EXPERT JUDGMENT]: Any other expression reflecting dissatisfaction, distrust, resistance, or critical views regarding the situation.

   - NEUTRAL: 
     * Objective news/Information sharing without opinion.
     * Unrelated to the specific case.

2. "keywords": 
   - Extract 5-8 MAIN WORDS/PHRASES (or fewer if the tweet is short) from the tweet that represent the topic.
   - Focus on:
     * Entities & Political Actors (e.g., "KPK", "Sudewo", "Partai", "DPRD")
     * Actions (e.g., "Tangkap", "Hukum Mati", "OTT")
     * Emotions (e.g., "Malu", "Kapok", "Geram")
     * Key Concepts/Objects (e.g., "Uang Rakyat", "Korupsi", "Pilkada", "Jalan Rusak").
     * [EXPERT JUDGMENT]: Other significant terms or slang that provide important context to the political narrative.
   - Normalize to lowercase and standard Indonesian if possible (e.g., convert "maling2" to "maling").
   - Format: A single string separated by comma (e.g., "korupsi, kpk, pati, hukum mati").

FORMAT OUTPUT (JSON LIST):
[
  {"id": "...", "sentiment": "...", "keywords": "..."},
  ...
]

NOTES:
- Be vigilant for SARCASM. Do not take apparent praise at face value.
- For keywords, keep it substantive. Ignore stopwords.
- If tweet is empty, return "NEUTRAL" and empty keywords.
"""

# api request function
def analyze_batch_retry(batch_data):
    try:
        payload = {
            "model": MODEL_ID,
            "messages": [
                {"role": "system", "content": system_prompt_final},
                {"role": "user", "content": json.dumps(batch_data)}
            ],
            "temperature": 0.3, "max_tokens": 2000, "reasoning": {"enabled": True} 
        }
        response = requests.post(
            url=URL, headers={"Authorization": f"Bearer {API_KEY}"}, data=json.dumps(payload), timeout=180
        )
        if response.status_code == 200:
            content = response.json()['choices'][0]['message']['content']
            clean_content = content.replace("```json", "").replace("```", "").strip()
            return json.loads(clean_content)
        return []
    except Exception as e:
        print(f"[!] error: {e}")
        return []

# main logic
# identify missing ids
all_ids = set(df_process['id'])
success_ids = set(df_llm['id'])
missing_ids = list(all_ids - success_ids)

print(f"[*] total missing data: {len(missing_ids)}")

# filter and run recovery
df_retry = df_process[df_process['id'].isin(missing_ids)][['id', 'text']]
retry_results = []

print("[*] starting recovery process...")
for i in tqdm(range(0, len(df_retry), 10)):
    batch_df = df_retry.iloc[i : i + 10].copy()
    batch_df['text'] = batch_df['text'].fillna("")
    
    batch_res = analyze_batch_retry(batch_df.to_dict(orient='records'))
    
    if batch_res:
        if isinstance(batch_res, list): retry_results.extend(batch_res)
        elif isinstance(batch_res, dict): 
            for v in batch_res.values(): 
                if isinstance(v, list): retry_results.extend(v)
    
    time.sleep(2)

# save recovery results
if retry_results:
    df_recover = pd.DataFrame(retry_results)
    df_recover.to_csv("output-data/recover_sentiment_bupati_pati.csv", index=False)
    print("\n[*] recovery file saved: recover_sentiment_bupati_pati.csv")

    # merge existing and recovered data
    df_full_sentiment = pd.concat([df_llm, df_recover], ignore_index=True)
    
    # rename columns to match final schema
    if 'sentiment' in df_full_sentiment.columns:
         df_full_sentiment.rename(columns={'sentiment': 'sentiment_llm', 'keywords': 'sentiment_keywords'}, inplace=True)
    
    # merge with main dataframe
    # remove old sentiment columns from process df if exists to avoid duplication
    cols = [c for c in df_process.columns if c not in ['sentiment_llm', 'sentiment_keywords']]
    df_final_ready = pd.merge(df_process[cols], df_full_sentiment, on='id', how='left')
    
    # save final complete file
    df_final_ready.to_csv("output-data/bupati_pati_FINAL_FULL.csv", index=False)
    print("[*] final complete file saved: bupati_pati_FINAL_FULL.csv")
    print(f"[*] remaining missing values: {df_final_ready['sentiment_llm'].isnull().sum()}")

else:
    print("[!] recovery failed or returned no data.")

[*] total missing data: 46
[*] starting recovery process...


100%|██████████| 5/5 [01:00<00:00, 12.02s/it]


[*] recovery file saved: recover_sentiment_bupati_pati.csv
[*] final complete file saved: bupati_pati_FINAL_FULL.csv
[*] remaining missing values: 0
